# Naive Bayes Classifier — SMS Spam Detection Demo

Naive Bayes is a family of **probabilistic classifiers** based on Bayes' theorem with a strong ("naive") assumption that features are conditionally independent given the class label. Despite this simplifying assumption, Naive Bayes classifiers perform remarkably well on text classification tasks.

## Bayes' Theorem

$$P(C \mid X) = \frac{P(X \mid C) \cdot P(C)}{P(X)}$$

Where:
- $P(C \mid X)$ — **posterior**: probability of class $C$ given features $X$
- $P(X \mid C)$ — **likelihood**: probability of observing features $X$ given class $C$
- $P(C)$ — **prior**: base probability of class $C$
- $P(X)$ — **evidence**: probability of observing features $X$ (constant across classes)

## The "Naive" Assumption

Naive Bayes assumes all features are **conditionally independent** given the class:

$$P(x_1, x_2, \ldots, x_n \mid C) = \prod_{i=1}^{n} P(x_i \mid C)$$

This is rarely true in practice (e.g. word co-occurrences in text), but the classifier still works well because it only needs to get the **ranking** of class probabilities right, not the exact values.

## Scikit-Learn Variants

| Variant | Feature Type | Best For |
|---------|-------------|----------|
| **MultinomialNB** | Word counts / TF-IDF | Text classification (standard choice) |
| **BernoulliNB** | Binary (word present/absent) | Short text, binary features |
| **ComplementNB** | Word counts / TF-IDF | Imbalanced datasets |

## Laplace Smoothing (alpha)

If a word never appears in training data for a given class, its probability becomes zero, making the entire product zero. **Laplace smoothing** adds a small count $\alpha$ to every feature:

$$P(x_i \mid C) = \frac{\text{count}(x_i, C) + \alpha}{\text{count}(C) + \alpha \cdot n}$$

- $\alpha = 1$: full Laplace smoothing (default)
- $\alpha < 1$: less smoothing, more trust in observed frequencies
- $\alpha > 1$: heavier smoothing

---

In this notebook we will:
1. Load and explore the SMS Spam dataset (5572 messages labelled ham or spam)
2. Preprocess text using TF-IDF vectorization
3. Train and evaluate a MultinomialNB baseline
4. Compare all three NB variants head-to-head
5. Investigate the effect of the alpha smoothing parameter
6. Tune hyperparameters with GridSearchCV
7. Inspect the most predictive words for each class
8. Make custom predictions on new messages

## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from data_loader import (
    load_spam_data,
    split_features_target,
    encode_target,
    vectorize_text,
    add_text_stats,
    get_class_names,
)
from model import (
    train_naive_bayes,
    evaluate_model,
    predict,
    compare_variants,
    cross_validate_model,
    alpha_sensitivity,
    tune_hyperparameters,
)
from visualization import (
    plot_class_distribution,
    plot_message_length_distribution,
    plot_word_count_distribution,
    plot_confusion_matrix,
    plot_roc_curve,
    plot_top_features,
    plot_variant_comparison,
    plot_alpha_tuning,
)

sns.set_theme(style="whitegrid", palette="Set2")
RANDOM_STATE = 42

%matplotlib inline

## 2. Load & Explore Data

The SMS Spam Collection dataset contains 5,572 SMS messages in English, each tagged as **ham** (legitimate) or **spam**. The dataset is notably imbalanced — roughly 87% ham and 13% spam.

In [ ]:
df = load_spam_data()
X, y = split_features_target(df)

print(f"Dataset shape : {df.shape}")
print(f"Class counts  : {y.value_counts().to_dict()}")
print(f"Spam ratio    : {(y == 'spam').mean():.1%}")
print()
df.head(10)

In [ ]:
print("--- Sample Ham Messages ---")
for msg in df[df["label"] == "ham"]["message"].sample(3, random_state=RANDOM_STATE).values:
    print(f"  {msg[:100]}")

print("\n--- Sample Spam Messages ---")
for msg in df[df["label"] == "spam"]["message"].sample(3, random_state=RANDOM_STATE).values:
    print(f"  {msg[:100]}")

## 3. Exploratory Data Analysis

In [ ]:
plot_class_distribution(y)
plt.show()

In [ ]:
df_stats = add_text_stats(df)
df_stats.groupby("label")[["message_length", "word_count"]].describe().round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_message_length_distribution(df_stats, ax=axes[0])
plot_word_count_distribution(df_stats, ax=axes[1])
plt.show()

Spam messages tend to be **longer** than ham messages — they need more words to convey their scam/offer. This length difference alone is a useful signal, but the specific words used carry far more discriminative power.

## 4. Text Preprocessing — TF-IDF Vectorization

Raw text cannot be fed directly into a classifier. We convert each message into a numeric vector using **TF-IDF** (Term Frequency–Inverse Document Frequency):

$$\text{tfidf}(t, d) = \text{tf}(t, d) \times \log\frac{N}{\text{df}(t)}$$

- **TF**: how often a term appears in this document (message)
- **IDF**: downweights terms that appear in many documents (common words like "the")

We also:
- Remove English stop words
- Use unigrams + bigrams (`ngram_range=(1, 2)`)
- Limit vocabulary to the top 5000 features
- Apply sublinear TF scaling (`1 + log(tf)`) to dampen the effect of very frequent terms

In [ ]:
y_enc, encoder = encode_target(y)
print(f"Label mapping: {dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, stratify=y_enc, random_state=RANDOM_STATE
)
print(f"\nTraining set : {len(X_train)} messages")
print(f"Test set     : {len(X_test)} messages")

In [ ]:
X_train_tfidf, X_test_tfidf, vectorizer = vectorize_text(X_train, X_test)

print(f"Vocabulary size      : {len(vectorizer.vocabulary_)} terms")
print(f"TF-IDF matrix (train): {X_train_tfidf.shape}")
print(f"TF-IDF matrix (test) : {X_test_tfidf.shape}")
print(f"Sparsity             : {1 - X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1]):.2%}")

## 5. Baseline Model — MultinomialNB

MultinomialNB is the standard choice for text classification with TF-IDF or word-count features. We start with the default smoothing parameter `alpha=1.0`.

In [ ]:
baseline = train_naive_bayes(X_train_tfidf, y_train, variant="multinomial", alpha=1.0)
baseline_results = evaluate_model(baseline, X_test_tfidf, y_test)

print(f"MultinomialNB (alpha=1.0)")
print(f"  Accuracy : {baseline_results['accuracy']:.4f}")
print(f"  Precision: {baseline_results['precision']:.4f}")
print(f"  Recall   : {baseline_results['recall']:.4f}")
print(f"  F1 Score : {baseline_results['f1']:.4f}")
print(f"\n{baseline_results['report']}")

## 6. Cross-Validation

A single train/test split can be misleading. Let's verify stability with 5-fold cross-validation.

In [ ]:
X_all_tfidf, _, vec_all = vectorize_text(X)
cv_results = cross_validate_model(X_all_tfidf, y_enc, variant="multinomial", cv=5)

print(f"5-Fold CV Accuracy: {cv_results['mean']:.4f} (+/- {cv_results['std']:.4f})")
print(f"Per-fold scores   : {[f'{s:.4f}' for s in cv_results['scores']]}")

## 7. Compare NB Variants

We train all three scikit-learn Naive Bayes variants on the same data and compare their performance:

- **MultinomialNB**: models word frequencies — the standard for text
- **BernoulliNB**: models binary word presence/absence — can work well for short texts
- **ComplementNB**: uses complement of each class to estimate parameters — designed for imbalanced data

In [ ]:
variant_results = compare_variants(X_train_tfidf, y_train, X_test_tfidf, y_test)

comparison_df = pd.DataFrame({
    name: {m: res[m] for m in ["accuracy", "precision", "recall", "f1"]}
    for name, res in variant_results.items()
}).T
comparison_df.index.name = "Variant"
comparison_df.round(4)

In [ ]:
plot_variant_comparison(variant_results)
plt.show()

## 8. Effect of Alpha (Smoothing Parameter)

The `alpha` parameter controls **Laplace smoothing**. We sweep across a log-scale range to see how it affects accuracy.

- Very small alpha: the model trusts raw frequency counts, risking zero-probability issues
- Very large alpha: heavy smoothing flattens probability differences between words

In [ ]:
alpha_scores = alpha_sensitivity(
    X_train_tfidf, y_train, X_test_tfidf, y_test,
    variant="multinomial"
)

best_alpha = max(alpha_scores, key=alpha_scores.get)
print(f"Best alpha: {best_alpha:.4f} (accuracy: {alpha_scores[best_alpha]:.4f})")

In [ ]:
plot_alpha_tuning(alpha_scores)
plt.show()

## 9. Hyperparameter Tuning with GridSearchCV

We perform an exhaustive search over alpha values using 5-fold cross-validation to find the optimal smoothing parameter for MultinomialNB.

In [ ]:
search = tune_hyperparameters(X_train_tfidf, y_train, variant="multinomial", cv=5)

print(f"Best parameters : {search.best_params_}")
print(f"Best CV accuracy: {search.best_score_:.4f}")

In [ ]:
cv_df = pd.DataFrame(search.cv_results_)
top_10 = cv_df.nsmallest(10, "rank_test_score")[
    ["params", "mean_test_score", "std_test_score", "rank_test_score"]
]
top_10

## 10. Final Evaluation

Evaluate the best model from GridSearchCV on the held-out test set.

In [ ]:
best_model = search.best_estimator_
best_results = evaluate_model(best_model, X_test_tfidf, y_test)

print(f"Best MultinomialNB — Test Results")
print(f"  Alpha    : {search.best_params_['alpha']:.4f}")
print(f"  Accuracy : {best_results['accuracy']:.4f}")
print(f"  Precision: {best_results['precision']:.4f}")
print(f"  Recall   : {best_results['recall']:.4f}")
print(f"  F1 Score : {best_results['f1']:.4f}")
print(f"\nClassification Report:\n{best_results['report']}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_confusion_matrix(best_results["confusion_matrix"],
                      class_names=get_class_names(), ax=axes[0])
plot_roc_curve(best_model, X_test_tfidf, y_test, ax=axes[1])
plt.show()

## 11. Top Predictive Words

One of the advantages of Naive Bayes is **interpretability** — we can inspect which words have the highest log-probability for each class. This reveals what the model has learned about spam vs ham language.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_top_features(vectorizer, best_model, n=15, ax=axes)
plt.show()

## 12. Custom Prediction

Let's classify some new messages. The pipeline is: raw text -> TF-IDF transform (using the fitted vectorizer) -> model prediction.

In [ ]:
custom_messages = [
    "Hey, are we still meeting for lunch tomorrow?",
    "CONGRATULATIONS! You've won a FREE iPhone! Click here to claim NOW!",
    "Reminder: your dentist appointment is at 3pm on Thursday.",
    "URGENT: Your account has been compromised. Call 0800-SCAM to verify.",
    "Can you pick up some milk on the way home?",
    "Win cash prizes! Text WIN to 80085 for your chance. Ts&Cs apply.",
]

custom_tfidf = vectorizer.transform(custom_messages)
predictions = predict(best_model, custom_tfidf)
predicted_labels = encoder.inverse_transform(predictions)

probas = best_model.predict_proba(custom_tfidf)

results_df = pd.DataFrame({
    "Message": [m[:60] + "..." if len(m) > 60 else m for m in custom_messages],
    "Prediction": predicted_labels,
    "P(ham)": probas[:, 0].round(4),
    "P(spam)": probas[:, 1].round(4),
})
print("Custom Predictions:")
results_df

## 13. Conclusion

**Key Takeaways**

1. **Naive Bayes is fast and effective for text classification** — training on 4,400+ messages with a 5,000-feature TF-IDF vocabulary takes milliseconds, yet achieves high accuracy.

2. **The "naive" independence assumption works surprisingly well** — even though words in natural language are clearly not independent, the classifier only needs correct class *rankings*, not calibrated probabilities.

3. **TF-IDF preprocessing is critical** — raw word counts overweight common words; TF-IDF downweights them via the IDF term, improving discriminative power. Stop word removal and n-grams further help.

4. **MultinomialNB is the standard for text** — it models word frequency distributions and consistently performs well. ComplementNB can help with class imbalance, while BernoulliNB works with binary features.

5. **Alpha (smoothing) matters** — too little smoothing risks zero-probability problems; too much flattens the model's ability to discriminate. The optimal alpha is typically found through cross-validation.

6. **Interpretability is a strength** — unlike neural networks, we can inspect the log-probabilities to see exactly which words drive predictions (e.g. "free", "win", "call" for spam).

7. **Class imbalance** — with ~87% ham and ~13% spam, precision on the spam class (avoiding false positives) is especially important. A legitimate message flagged as spam is worse than a spam message reaching the inbox.

**Next Steps**
- Compare with other classifiers (Logistic Regression, SVM, Random Forest)
- Experiment with different text preprocessing (stemming, lemmatization, character n-grams)
- Try on other text classification datasets (sentiment analysis, topic classification)
- Build a simple pipeline with `sklearn.pipeline.Pipeline` for production use